# DATA 298B — Colab Live Dashboard Demo

This notebook prepares the existing React/Vite dashboard for a Colab runtime demo. It mounts the existing `DATA298B` Drive structure, clones `feature/298b-backend`, and starts the frontend shown in the final presentation screenshot.

The normal dashboard demo uses saved XGBoost and LLM report artifacts. The optional Gemma cell at the end performs one live adapter inference only when explicitly enabled.

In [ ]:
# Configuration — run this notebook from top to bottom.
REPO_URL = 'https://github.com/aryaMehta26/Graph-Based-Fraud-Intelligence-Platform.git'
REPO_BRANCH = 'feature/298b-live-dashboard'
DRIVE_ROOT = '/content/drive/MyDrive/DATA298B'
REPO_DIR = '/content/Graph-Based-Fraud-Intelligence-Platform'
FRONTEND_PORT = 4173
RUN_LIVE_GEMMA = False  # Change to True only for an explicit one-case GPU smoke test.
GEMMA_ADAPTER_DIR = f'{DRIVE_ROOT}/adapters/gemma_full_grounded'
# If your Drive adapter has a different final name, change only this path.
GEMMA_MODEL_ID = 'google/gemma-4-31B'

In [ ]:
# Mount the existing shared Drive structure.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, sys

repo_path = Path(REPO_DIR)
if repo_path.exists():
    subprocess.run(['git', '-C', str(repo_path), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo_path), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo_path), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
os.chdir(repo_path)
sys.path.insert(0, str(repo_path))
os.environ['DATA298B_LIVE_GEMMA'] = '1' if RUN_LIVE_GEMMA else '0'
os.environ['DATA298B_GEMMA_MODEL_ID'] = GEMMA_MODEL_ID
os.environ['DATA298B_GEMMA_ADAPTER_DIR'] = GEMMA_ADAPTER_DIR
os.environ['DATA298B_GEMMA_TRACE_DIR'] = f'{DRIVE_ROOT}/artifacts/live_gemma_traces'
print('Repository:', repo_path)
print('Branch:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())

In [ ]:
# Install the dashboard/runtime dependencies.
%pip install -q pandas pyarrow xgboost scikit-learn streamlit pyvis neo4j python-dotenv pyyaml transformers peft accelerate bitsandbytes sentencepiece

# Confirm the runtime and GPU. XGBoost/dashboard can run without a GPU; live Gemma cannot.
import torch
print('Torch:', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
else:
    print('No CUDA GPU detected. The dashboard demo can still run; leave RUN_LIVE_GEMMA=False.')

In [ ]:
# Link the existing Drive input artifacts into the paths used by the repository snapshot tooling.
# This does not copy the multi-million-row Parquet files or modify the Drive data.
from pathlib import Path

drive_input = Path(DRIVE_ROOT) / 'input'
processed = Path(REPO_DIR) / 'data' / 'processed'
processed.mkdir(parents=True, exist_ok=True)

def link_drive_file(source_name, target_name):
    source = drive_input / source_name
    target = processed / target_name
    if not source.exists():
        print('Missing:', source)
        return False
    if target.exists() or target.is_symlink():
        target.unlink()
    target.symlink_to(source)
    print(f'{target.name} -> {source}')
    return True

# The current Drive layout contains the graph-enriched train/test files.
link_drive_file('test_graph_enriched.parquet', 'test_graph_enriched.parquet')
link_drive_file('test_graph_enriched.parquet', 'split_test.parquet')
link_drive_file('train_graph_enriched.parquet', 'train_graph_enriched.parquet')

# Check the row count without loading the full Parquet file into RAM.
import pyarrow.parquet as pq
test_path = processed / 'test_graph_enriched.parquet'
if test_path.exists():
    print('Test rows:', pq.ParquetFile(test_path).metadata.num_rows)
    print('Columns:', pq.ParquetFile(test_path).schema.names[:20], '...')

In [ ]:
# Verify the repository and saved artifacts before starting the React dashboard.
from pathlib import Path
required = [
    Path(REPO_DIR) / 'frontend' / 'package.json',
    Path(REPO_DIR) / 'data/models/xgboost_baseline.json',
    Path(REPO_DIR) / 'data/models/xgboost_graph_enhanced.json',
    Path(REPO_DIR) / 'data/processed/test_graph_enriched.parquet',
]
for path in required:
    print(('OK  ' if path.exists() else 'MISS'), path)

print('Drive adapter path:', GEMMA_ADAPTER_DIR)
print('Adapter exists:', Path(GEMMA_ADAPTER_DIR).exists())

In [ ]:
# Start the live Python bridge: replayed XGBoost predictions plus optional Gemma inference.
import os, subprocess, time
subprocess.run(['pkill', '-f', 'live_dashboard_api.py'], check=False)
api_log_path = '/content/live_dashboard_api.log'
api_log = open(api_log_path, 'w')
api_process = subprocess.Popen(
    ['python', 'scripts/live_dashboard_api.py'],
    cwd=REPO_DIR,
    stdout=api_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)
time.sleep(8)
print('Live API PID:', api_process.pid)
print(Path(api_log_path).read_text()[-3000:])

In [ ]:
# Install and start the React/Vite dashboard shown in the final UI.
import subprocess, time
frontend_dir = Path(REPO_DIR) / 'frontend'
# Allow the temporary ngrok hostname used for the public demo URL.
os.environ['__VITE_ADDITIONAL_SERVER_ALLOWED_HOSTS'] = '.ngrok-free.dev,.ngrok.io'
subprocess.run(['npm', 'install', '--silent'], cwd=frontend_dir, check=True)
log_path = '/content/frontend_dashboard.log'
log_file = open(log_path, 'w')
frontend_process = subprocess.Popen([
    'npm', 'run', 'dev', '--',
    '--host', '0.0.0.0',
    '--port', str(FRONTEND_PORT),
], stdout=log_file, stderr=subprocess.STDOUT, cwd=frontend_dir)
time.sleep(8)
print('Frontend PID:', frontend_process.pid)
print('Log:', log_path)
print(Path(log_path).read_text()[-2000:])

In [ ]:
# Open the React dashboard inside the notebook using Colab's supported iframe helper.
from google.colab import output
output.serve_kernel_port_as_iframe(FRONTEND_PORT, height=900)

# Optional public link for teammates. Add a Colab Secret named NGROK_AUTHTOKEN first.
%pip install -q pyngrok
from google.colab import userdata
from pyngrok import ngrok

try:
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    ngrok_token = ''

if ngrok_token:
    ngrok.kill()
    ngrok.set_auth_token(ngrok_token)
    public_url = ngrok.connect(FRONTEND_PORT, 'http').public_url
    print('Public dashboard URL:', public_url)
else:
    print('NGROK_AUTHTOKEN was not found in Colab Secrets; using the embedded iframe only.')

## Optional: one live Gemma smoke test

The React dashboard uses the checked-in DATA 298B snapshot for the reference pages, while the live API adds replayed XGBoost predictions and optional Gemma investigation requests.

In [ ]:
print('The live API and React UI are running.')
print('Overview polls live XGBoost predictions; LLM Investigator can call Gemma when RUN_LIVE_GEMMA=True.')

## Demo sequence

1. Set `RUN_LIVE_GEMMA = True` if you want live Gemma reports, then run the setup cells.
2. Open **Overview** to show the live replayed XGBoost prediction stream.
3. Open **Transactions** to show transaction context from the saved snapshot.
4. Open **Network Analysis** to show graph/community evidence.
5. Open **LLM Investigator** and click **Run live Gemma report** for a new structured report.
6. Use **Reports** to show the saved validated reports alongside the live investigation.

This is a Colab-hosted demo runtime. Closing or timing out the runtime stops the dashboard and any live model process.